# Q-Mini training pipeline
From-scratch, attention-free CausalConv-MoE model. No pretrained weights are downloaded. Keep datasets and checkpoints in Drive, not this public source repository.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA = Path('/content/drive/MyDrive/QMini/datasets')
ART = Path('/content/drive/MyDrive/QMini/artifacts')
ART.mkdir(parents=True, exist_ok=True)
print('Data:', DATA, '| Artifacts:', ART)


In [ ]:
import os, subprocess
REPO = Path('/content/Q-Mini')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/lion36508/Q-Mini.git',str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)
subprocess.run(['python','-m','pip','install','-e','.[dev]'],check=True)
import torch
print('PyTorch',torch.__version__,'CUDA',torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


In [ ]:
TOK = ART/'tokenizer'/'tokenizer.json'; CFG = ART/'qmini_active_config.json'
TOK.parent.mkdir(parents=True, exist_ok=True)
if not TOK.exists():
    subprocess.run([
        'python','-m','qmini.cli','tokenizer',
        '--data-dir',str(DATA),'--output',str(TOK),
        '--vocab-size','4096','--valid-fraction','0.02',
        '--config-output',str(CFG)
    ],check=True)
else:
    print('Existing tokenizer preserved:', TOK)
if not CFG.is_file():
    raise FileNotFoundError(
        'Tokenizer exists but its active config is missing. Restore the matching config; '
        'do not silently retrain or overwrite the tokenizer.'
    )

# Create a genuinely small smoke model but retain the exact tokenizer vocabulary.
# The stock smoke config has a 128-token vocabulary and must never be paired with
# the full-corpus token streams.
from qmini.config import QMiniConfig
base_config = QMiniConfig.from_json(CFG)
smoke_values = base_config.to_dict()
smoke_values.update({
    'd_model': 32,
    'd_hidden': 48,
    'n_layers': 2,
    'n_experts': 2,
    'top_k': 1,
    'max_context': 64,
    'kernel_size': 5,
    'dilations': [1, 2],
    'max_thinking_steps': 2,
    'thinking_steps': 1,
})
smoke_config = QMiniConfig.from_dict(smoke_values)
SMOKE_CFG = ART/'qmini_smoke_active_config.json'
smoke_config.save(SMOKE_CFG)
print('Base vocabulary:', base_config.vocab_size)
print('Smoke vocabulary:', smoke_config.vocab_size)
print('Smoke model parameters are computed by the runtime, not hard-coded.')


In [ ]:
TOKENS = ART/'tokens_v1'
required_artifacts = [
    TOKENS/'train.bin',
    TOKENS/'valid.bin',
    TOKENS/'train.mask.bin',
    TOKENS/'valid.mask.bin',
    TOKENS/'dataset.json',
]
present = [item.is_file() for item in required_artifacts]
if any(present) and not all(present):
    raise RuntimeError(
        'An incomplete token artifact set already exists. It has been preserved. '
        'Set TOKENS to a new versioned directory such as tokens_v2; do not mix old and new streams.'
    )
if not all(present):
    subprocess.run([
        'python','-m','qmini.cli','prepare',
        '--data-dir',str(DATA),'--tokenizer',str(TOK),
        '--output-dir',str(TOKENS),'--valid-fraction','0.02',
        '--loss-mask','auto'
    ],check=True)
else:
    print('Existing token streams preserved:', TOKENS)

# Verify that token IDs, loss masks and tokenizer all belong to one preparation run.
import hashlib, json
manifest = json.loads((TOKENS/'dataset.json').read_text(encoding='utf-8'))
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
if manifest.get('tokenizer_sha256', {}).get('sha256') != sha256_file(TOK):
    raise RuntimeError('Tokenizer hash does not match token-stream manifest.')
for stream_name, mask_name in [
    ('train.bin','train.mask.bin'),
    ('valid.bin','valid.mask.bin'),
]:
    stream = TOKENS/stream_name
    mask = TOKENS/mask_name
    if mask.stat().st_size != stream.stat().st_size // 2:
        raise RuntimeError(f'Loss-mask length mismatch for {stream_name}.')
print('train tokens:', (TOKENS/'train.bin').stat().st_size//2)
print('valid tokens:', (TOKENS/'valid.bin').stat().st_size//2)
print('train documents:', manifest['train_documents'])
print('valid documents:', manifest['valid_documents'])
print('loss-mask mode:', manifest['loss_mask_mode'])


In [ ]:
# Use a fresh, versioned smoke run. Earlier runs may have been created with an
# incompatible 128-token vocabulary; keep them untouched.
SMOKE = ART/'runs'/'smoke_v2'
smoke_args = [
    'python','-m','qmini.cli','train',
    '--config',str(SMOKE_CFG),
    '--train-bin',str(TOKENS/'train.bin'),
    '--valid-bin',str(TOKENS/'valid.bin'),
    '--train-mask-bin',str(TOKENS/'train.mask.bin'),
    '--valid-mask-bin',str(TOKENS/'valid.mask.bin'),
    '--tokenizer',str(TOK),
    '--output-dir',str(SMOKE),
    '--steps','5','--batch-size','1','--seq-len','32','--grad-accum','1',
    '--eval-every','5','--save-every','5','--eval-batches','2'
]
checkpoint_path = SMOKE/'latest.pt'
if checkpoint_path.is_file():
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    if checkpoint.get('config') != smoke_config.to_dict():
        raise RuntimeError(
            'Existing smoke checkpoint config differs. It was preserved; choose a new smoke run directory.'
        )
    if int(checkpoint.get('step', 0)) < 5:
        smoke_args.append('--resume')
        subprocess.run(smoke_args, check=True)
    else:
        print('Compatible smoke run already completed; checkpoint preserved:', checkpoint_path)
else:
    subprocess.run(smoke_args, check=True)


In [ ]:
BASE = ART/'runs'/'base'
args = ['python','-m','qmini.cli','train','--config',str(CFG),'--train-bin',str(TOKENS/'train.bin'),'--valid-bin',str(TOKENS/'valid.bin'),'--train-mask-bin',str(TOKENS/'train.mask.bin'),'--valid-mask-bin',str(TOKENS/'valid.mask.bin'),'--tokenizer',str(TOK),'--output-dir',str(BASE),'--steps','10000','--batch-size','2','--seq-len','512','--grad-accum','8','--eval-every','250','--save-every','250']
if (BASE/'latest.pt').exists(): args.append('--resume')
subprocess.run(args,check=True)
